In [1]:
import warnings
import logging

warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

from litellm import completion

In [2]:
import litellm
litellm.suppress_debug_info = True

## Simplest LLM

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

print("OpenAI Key loaded", "yes" if os.getenv("OPENAI_API_KEY") else "OpenAI Key not found")
print("Groq API Key loaded", "yes" if os.getenv("GROQ_API_KEY") else "Groq API Key not found")

OpenAI Key loaded yes
Groq API Key loaded yes


In [4]:
from litellm import completion

response_openai = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)

print("openai response:", response_openai.choices[0].message.content)

openai response: RAG (Retrieve and Generate) is a framework that combines information retrieval and natural language generation, allowing models to first fetch relevant data from a database before generating coherent and contextually appropriate responses.


In [5]:
import os
import requests

response = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
    timeout=30,
)
response.raise_for_status()
print([model["id"] for model in response.json()["data"]])

['openai/gpt-oss-120b', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-20b', 'canopylabs/orpheus-v1-english', 'whisper-large-v3', 'whisper-large-v3-turbo', 'qwen/qwen3.8-27b', 'openai/gpt-oss-safeguard-20b', 'allam-2-7b', 'meta-llama/llama-prompt-guard-2-22m', 'canopylabs/orpheus-arabic-saudi']


In [6]:
response_groq = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}],
)

print("Groq response:", response_groq.choices[0].message.content)

Groq response: RAG (Retrieval‑Augmented Generation) is a technique that first pulls in relevant external documents and then feeds them, along with the prompt, into a language model so it can generate answers grounded in up‑to‑date, specific information.


## Automatic fallbacks

In [7]:
from litellm import completion
from litellm import Router

router = Router(
    model_list=[
        {
            "model_name": "primary",
            "litellm_params": {
                "model": "gpt-4o-mini",
                "api_key": "os.environ/OPENAI_API_KEY",
            },
        },
        {
            "model_name": "fallback",
            "litellm_params": {
                "model": "groq/openai/gpt-oss-20b",
                "api_key": "os.environ/GROQ_API_KEY",
            },
        },
    ],
    fallbacks=[{"primary": ["fallback"]}],
)

response = router.completion(
    model="primary",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}],
)

print("response:", response.choices[0].message.content[:200], "...")
print("which model was used:", response.model)

response: RAG (Retrieval-Augmented Generation) is a machine learning approach that combines the capabilities of information retrieval and natural language generation to enhance the quality and relevance of gene ...
which model was used: gpt-4o-mini-2024-07-18


In [8]:
from litellm import Router

router = Router(
    model_list=[
        {
            "model_name": "primary",
            "litellm_params": {
                "model": "openai/fake-nonexistent-model",
                "api_key": "os.environ/OPENAI_API_KEY",
            },
        },
        {
            "model_name": "groq-fallback",
            "litellm_params": {
                "model": "groq/openai/gpt-oss-20b",
                "api_key": "os.environ/GROQ_API_KEY",
            },
        },
    ],
    fallbacks=[{"primary": ["groq-fallback"]}],
)

response = router.completion(
    model="primary",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}],
)

print("Response:", response.choices[0].message.content[:200], "...")
print("Model reported by provider:", response.model)

Response: Retrieval‑augmented generation (RAG) is a technique that first pulls relevant external documents into a context window and then uses a language model to generate an answer conditioned on those retriev ...
Model reported by provider: openai/gpt-oss-20b


In [9]:
from litellm import completion, completion_cost

response = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}],
)

cost = completion_cost(completion_response=response)
usage = response.usage

print("Response:", response.choices[0].message.content[:200], "...")
print("Input tokens:", usage.prompt_tokens)
print("Output tokens:", usage.completion_tokens)
print(f"Cost: ${cost:.8f}")

Response: RAG, or Retrieval-Augmented Generation, is a natural language processing approach that combines traditional retrieval techniques with generative models to enhance the quality and relevance of generate ...
Input tokens: 14
Output tokens: 39
Cost: $0.00002550


## caching: don't pay twice for same 

In [10]:
import time
import litellm
from litellm.caching import Cache
from litellm import completion

litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm.cache = Cache(type="local")

prompt = "What does LLM stand for?"

start = time.time()
r1 = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    caching=True,
)
t1 = time.time() - start
print(f"First call: {t1:.2f}s - {r1.choices[0].message.content}")

start = time.time()
r2 = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    caching=True,
)
t2 = time.time() - start
print(f"Second call: {t2:.2f}s - {r2.choices[0].message.content}")

First call: 0.94s - LLM stands for "Large Language Model." It refers to a type of artificial intelligence model that is trained on vast amounts of text data to understand and generate human-like language. Examples of large language models include OpenAI's GPT-3 and GPT-4.
Second call: 0.00s - LLM stands for "Large Language Model." It refers to a type of artificial intelligence model that is trained on vast amounts of text data to understand and generate human-like language. Examples of large language models include OpenAI's GPT-3 and GPT-4.


## Smart routing: right model for right task

In [11]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",                              # 👈 alias kept
        "litellm_params": {
            "model": "gpt-4o",                                      # 👈 mapped to OpenAI instead
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    }
]

router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response = router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)

print("⚡ Fast/cheap (Groq): ", fast_response.choices[0].message.content[:150])
print("\n🧠 Smart/coding (GPT-4o):\n", code_response.choices[0].message.content[:300])

⚡ Fast/cheap (Groq):  **AI is reshaping software in three major waves**

| Wave | What’s happening | Why it matters |
|------|------------------|----------------|
| **1️⃣ C

🧠 Smart/coding (GPT-4o):
 Certainly! Here's a simple Python function to reverse a string:

```python
def reverse_string(input_string):
    """
    This function takes a string as input and returns the reversed string.

    :param input_string: str, the string to be reversed
    :return: str, the reversed string
    """
    r


## Load balancing

In [12]:
from litellm import Router
import os

# Two deployments under the same alias
# A pool of "smart" models — all equally capable, just different providers
model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "gpt-4o",
            "api_key": os.getenv("OPENAI_API_KEY"),
        },
        "model_info": {"id": "openai-gpt4o"}
    },

    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq-llama-70b"}
    },
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Say hello, request {i+1}"}]
    )
    # Pull out which deployment served this request
    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        openai-gpt4o             589 ms   Hello! How can I assist you today?
#2        openai-gpt4o             790 ms   Hello! How can I assist you today?
#3        openai-gpt4o             716 ms   Hello! How can I assist you today?
#4        openai-gpt4o             857 ms   Hello! How can I assist you today?
#5        groq-llama-70b           728 ms   Hello! Could you clarify what you’d
#6        openai-gpt4o             716 ms   Hello! How can I assist you today?


## Routing strategy: Least Busy

In [13]:
import os
from litellm import Router
from collections import Counter

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gpt-4o-mini",
                        "api_key": os.getenv("OPENAI_API_KEY")},
     "model_info": {"id": "🔵 OpenAI"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/openai/gpt-oss-20b",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="least-busy"   # 👈 the magic
)

hits = Counter()
for i in range(8):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": f"Say 'OK' #{i}"}],
        max_tokens=5
    )
    hits[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

print("\n🎯 Distribution:")
for k, v in hits.most_common():
    print(f"   {k}: {'█' * v} ({v})")

Request 1 → 🔵 OpenAI
Request 2 → 🔵 OpenAI
Request 3 → 🔵 OpenAI
Request 4 → 🔵 OpenAI
Request 5 → 🔵 OpenAI
Request 6 → 🔵 OpenAI
Request 7 → 🔵 OpenAI
Request 8 → 🔵 OpenAI

🎯 Distribution:
   🔵 OpenAI: ████████ (8)


## Integration with Langchain

In [16]:
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "langchain-litellm", "litellm"]
)

# Your original code follows below:
from langchain_litellm import ChatLiteLLM


  Using cached langchain_litellm-0.11.1-py3-none-any.whl.metadata (5.0 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-1.7.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached litellm-1.106.3-cp310-abi3-macosx_11_0_arm64.whl.metadata (45 kB)
  Using cached pydantic-2.14.1-py3-none-any.whl.metadata (133 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached filelock-3.32.7-py3-none-any.whl.metadata (2.0 kB)
  Using cached openai-2.54.0-py3-none-any.whl.metadata (40 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached aiohttp-3.14.5-cp312-cp312-macosx_11_0_arm64.whl.metadata (8.3 kB)
  Using cached pydantic_settings-2.16.1-py3-none-any.whl.metadata (4.0 kB)
  Using cached boto3-1.43.118-py3-none-any.whl.metadata (6.6 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached botocore-1.43.118-py3-none-any.whl.metadata (5.6 kB)
  Using cached jmespa

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-text-splitters 0.3.2 requires langchain-core<0.4.0,>=0.3.15, but you have langchain-core 1.7.0 which is incompatible.
langchain-community 0.3.5 requires langchain-core<0.4.0,>=0.3.15, but you have langchain-core 1.7.0 which is incompatible.
langchain-community 0.3.5 requires langsmith<0.2.0,>=0.1.125, but you have langsmith 0.14.12 which is incompatible.
langchain 0.3.7 requires langchain-core<0.4.0,>=0.3.15, but you have langchain-core 1.7.0 which is incompatible.
langchain 0.3.7 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.14.12 which is incompatible.


In [17]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Build a chat model that talks through LiteLLM
llm = ChatLiteLLM(model="gpt-4o-mini", temperature=0.3)

# A standard LangChain prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor named KrishGPT. Be concise."),
    ("user", "{question}")
])

# Compose with LCEL — same syntax as native LangChain
chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"question": "What is an LLM Gateway in 3 bullets?"})
print(answer)

- **Definition**: An LLM Gateway is an interface or platform that facilitates access to Large Language Models (LLMs) for various applications, enabling users to leverage their capabilities without needing extensive technical knowledge.

- **Functionality**: It typically provides APIs or user-friendly tools that allow developers and businesses to integrate LLMs into their products, services, or workflows, streamlining the process of utilizing advanced AI for tasks like text generation, summarization, and conversation.

- **Benefits**: LLM Gateways enhance accessibility, reduce implementation complexity, and often include features like scalability, security, and support, making it easier for organizations to adopt AI technologies effectively.
